In [1]:
##!pip install taxopy[fuzzy-matching]
import os

import taxopy

import pandas as pd

##!pip install pivottablejs
#from pivottablejs import pivot_ui

from rapidfuzz import process

import seaborn as sns

In [4]:
#wget ftp://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz
taxdb = taxopy.TaxDb(nodes_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/nodes.dmp", names_dmp="/home/dcm/250513Pat_250728Pat_250930Pat/taxdump/names.dmp")

In [5]:
#read culture_tax df and clean
df_culture_tax = pd.read_csv('culture/Culture_metadata_10_26_25_extra.txt', sep='\t')
#if isolate has multiple species IDs possible, assume it is a mixed isolate and separate each species ID all as new row
df_culture_tax = df_culture_tax.assign(Species=df_culture_tax['Species'].str.split(' / ')).explode('Species')
df_culture_tax= df_culture_tax[['Species']]
df_culture_tax = df_culture_tax.drop_duplicates()
df_culture_tax


#function to compare tax name found by taxopy versus input name and pick best match
def best_match(target, choices_dict):
    values = list(choices_dict.values())
    match_value, score, _ = process.extractOne(target, values)

    for key, value in choices_dict.items():
        if value == match_value:
            return {key: (match_value, score)}

    return {}  # return empty dict if no match

#loop over species names, get taxid and matching species name from taxopy
#then get closest match to query
#then get full tax lineage
#append results to output df

df = pd.DataFrame(columns=['Species','best_taxid','best_match_score','phylum','class','order','family','genus', 'species'])

for Species in df_culture_tax['Species']:

    try:
        taxids = taxopy.taxid_from_name(f"{Species}", taxdb, fuzzy=True, score_cutoff=0.80) #set threshold lower to capture missed taxa
        tax_dict = {}

        for taxid in taxids:
            tax_name = taxopy.Taxon(taxid, taxdb).name
            tax_dict[taxid] = tax_name

        best_tax_match = best_match(f"{Species}", tax_dict)
        best_taxid, best_taxname_score = next(iter(best_tax_match.items()))
        best_taxname = best_taxname_score[0]
        best_match_score = best_taxname_score[1]

        taxa_dict = taxopy.Taxon(best_taxid, taxdb).rank_name_dictionary
        #Append only the selected items
        selected_data = {key: taxa_dict.get(key, 'N/A') for key in ['Species','best_taxid','best_match_score','phylum','class','order','family','genus', 'species']}
        selected_data['Species'] = f"{Species}"
        selected_data['best_taxid'] = best_taxid
        selected_data['best_match_score'] = best_match_score
        df = df._append(selected_data, ignore_index=True)
        
    except Exception as e:
        print(f"An error occurred for Species '{Species}': {e}")
        continue

/home/dcm/env/lib/python3.8/site-packages/taxopy/utilities.py:151: Warning: The input name was not found in the taxonomy database.
  warnings.warn(


An error occurred for Species 'Ochrobactrum pseudogrignonense': cannot unpack non-iterable NoneType object


/home/dcm/env/lib/python3.8/site-packages/taxopy/utilities.py:151: Warning: The input name was not found in the taxonomy database.
  warnings.warn(


An error occurred for Species 'Gemella sp. strain KCOM 1824': cannot unpack non-iterable NoneType object


/home/dcm/env/lib/python3.8/site-packages/taxopy/utilities.py:151: Warning: The input name was not found in the taxonomy database.
  warnings.warn(


An error occurred for Species 'Bacillus circulans': cannot unpack non-iterable NoneType object


In [6]:
#read culture_tax df and clean
df_culture_tax = pd.read_csv('culture/Culture_metadata_11-2-25_mods.txt', sep='\t')
#if isolate has multiple species IDs possible, assume it is a mixed isolate and separate each species ID all as new row
df_culture_tax = df_culture_tax.assign(Species=df_culture_tax['Species'].str.split(' / ')).explode('Species')
df_culture_tax= df_culture_tax[['Species']]
df_culture_tax = df_culture_tax.drop_duplicates()
df_culture_tax


#function to compare tax name found by taxopy versus input name and pick best match
def best_match(target, choices_dict):
    values = list(choices_dict.values())
    match_value, score, _ = process.extractOne(target, values)

    for key, value in choices_dict.items():
        if value == match_value:
            return {key: (match_value, score)}

    return {}  # return empty dict if no match

#loop over species names, get taxid and matching species name from taxopy
#then get closest match to query
#then get full tax lineage
#append results to output df

df = pd.DataFrame(columns=['Species','best_taxid','best_match_score','phylum','class','order','family','genus', 'species'])

for Species in df_culture_tax['Species']:

    try:
        taxids = taxopy.taxid_from_name(f"{Species}", taxdb, fuzzy=True, score_cutoff=0.80) #set threshold lower to capture missed taxa
        tax_dict = {}

        for taxid in taxids:
            tax_name = taxopy.Taxon(taxid, taxdb).name
            tax_dict[taxid] = tax_name

        best_tax_match = best_match(f"{Species}", tax_dict)
        best_taxid, best_taxname_score = next(iter(best_tax_match.items()))
        best_taxname = best_taxname_score[0]
        best_match_score = best_taxname_score[1]

        taxa_dict = taxopy.Taxon(best_taxid, taxdb).rank_name_dictionary
        #Append only the selected items
        selected_data = {key: taxa_dict.get(key, 'N/A') for key in ['Species','best_taxid','best_match_score','phylum','class','order','family','genus', 'species']}
        selected_data['Species'] = f"{Species}"
        selected_data['best_taxid'] = best_taxid
        selected_data['best_match_score'] = best_match_score
        df = df._append(selected_data, ignore_index=True)
        
    except Exception as e:
        print(f"An error occurred for Species '{Species}': {e}")
        continue

In [7]:
df

,Species,best_taxid,best_match_score,phylum,class,order,family,genus,species
0,Cryptobacterium curtum,84163,100.0,Actinomycetota,Coriobacteriia,Eggerthellales,Eggerthellaceae,Cryptobacterium,Cryptobacterium curtum
1,Lancefieldella parvula,1382,100.0,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Lancefieldella,Lancefieldella parvula
2,Ligilactobacillus salivarius,1624,100.0,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Ligilactobacillus,Ligilactobacillus salivarius
3,Rothia dentocariosa,2047,100.0,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia dentocariosa
4,Rothia mucilaginosa,43675,100.0,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia mucilaginosa
...,...,...,...,...,...,...,...,...,...
183,Gemella taiwanensis,1179787,100.0,Bacillota,Bacilli,Bacillales,Gemellaceae,Gemella,Gemella taiwanensis
184,Lachnoanaerobaculum umeaense,617123,100.0,Bacillota,Clostridia,Lachnospirales,Lachnospiraceae,Lachnoanaerobaculum,Lachnoanaerobaculum umeaense
185,Veillonella tobetsuensis,1110546,100.0,Bacillota,Negativicutes,Veillonellales,Veillonellaceae,Veillonella,Veillonella tobetsuensis
186,Actinomyces sp. oral strain,135018,95.0,Actinomycetota,Actinomycetes,Actinomycetales,Actinomycetaceae,Actinomyces,Actinomyces sp. oral strain B19SC


In [8]:
#read culture_tax df and clean
df_culture_tax = pd.read_csv('culture/Culture_metadata_11-2-25_mods.txt', sep='\t')
#if isolate has multiple species IDs possible, assume it is a mixed isolate and separate each species ID all as new row
df_culture_tax = df_culture_tax.assign(Species=df_culture_tax['Species'].str.split(' / ')).explode('Species')

dfm = pd.merge(df_culture_tax, df, on = 'Species', how ='left')

dfm.to_csv("culture/Culture_metadata_11-2-25_mods_taxids.txt", sep='\t', index = False)

dfm

,MIT_Accession,Subject ID,Vanderbilt_ID,Tissue,Biopsy_collection_date_year,Batch,Shipment date,Sex,Age at baseline,Prog-Nonprog between 20-26y,...,Genus,Notes,best_taxid,best_match_score,phylum,class,order,family,genus,species
0,23-1062,4100,NQ4828,"antrum, greater curvature",20,1,11/6/2023,M,53,P,...,Cryptobacterium,NaN,84163,100.0,Actinomycetota,Coriobacteriia,Eggerthellales,Eggerthellaceae,Cryptobacterium,Cryptobacterium curtum
1,23-1062,4100,NQ4828,"antrum, greater curvature",20,1,11/6/2023,M,53,P,...,Lancefieldella,NaN,1382,100.0,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Lancefieldella,Lancefieldella parvula
2,23-1062,4100,NQ4828,"antrum, greater curvature",20,1,11/6/2023,M,53,P,...,Ligilactobacillus,NaN,1624,100.0,Bacillota,Bacilli,Lactobacillales,Lactobacillaceae,Ligilactobacillus,Ligilactobacillus salivarius
3,23-1062,4100,NQ4828,"antrum, greater curvature",20,1,11/6/2023,M,53,P,...,Rothia,NaN,2047,100.0,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia dentocariosa
4,23-1062,4100,NQ4828,"antrum, greater curvature",20,1,11/6/2023,M,53,P,...,Rothia,NaN,43675,100.0,Actinomycetota,Actinomycetes,Micrococcales,Micrococcaceae,Rothia,Rothia mucilaginosa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,24-0133,9186,NQ5352,"antrum, greater curvature",26,3,6/10/2024,M,40,P,...,Streptococcus,NaN,1302,100.0,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,Streptococcus gordonii
1015,24-0133,9186,NQ5352,"antrum, greater curvature",26,3,6/10/2024,M,40,P,...,Streptococcus,NaN,28037,100.0,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,Streptococcus mitis
1016,24-0133,9186,NQ5352,"antrum, greater curvature",26,3,6/10/2024,M,40,P,...,Streptococcus,NaN,1234680,100.0,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,Streptococcus rubneri
1017,24-0133,9186,NQ5352,"antrum, greater curvature",26,3,6/10/2024,M,40,P,...,Streptococcus,NaN,1304,100.0,Bacillota,Bacilli,Lactobacillales,Streptococcaceae,Streptococcus,Streptococcus salivarius


In [9]:
df = pd.read_csv("culture/Culture_metadata_11-2-25_mods_taxids.txt", sep='\t')
pivot_counts = df.pivot_table(index='genus', columns='MIT_Accession', aggfunc='size')

# Fill NaN values in the pivot table with 0
pivot_counts = pivot_counts.fillna(0).reset_index()
pivot_counts.to_csv("culture/genus_otus.txt", sep = '\t', index = False)
pivot_counts = df[['genus']].copy()     # subset to genus
pivot_counts['Genus'] = pivot_counts['genus']
pivot_counts = pivot_counts.drop_duplicates() 
pivot_counts.to_csv("culture/genus_tax.txt", sep="\t", index=False)


df = pd.read_csv("culture/Culture_metadata_11-2-25_mods_taxids.txt", sep='\t')
pivot_counts = df.pivot_table(index='species', columns='MIT_Accession', aggfunc='size')

# Fill NaN values in the pivot table with 0
pivot_counts = pivot_counts.fillna(0).reset_index()
pivot_counts.to_csv("culture/species_otus.txt", sep = '\t', index = False)

pivot_counts = df[['species']].copy()     # subset to genus
pivot_counts['Species'] = pivot_counts['species']   # duplicate the column
pivot_counts = pivot_counts.drop_duplicates() 
pivot_counts.to_csv("culture/species_tax.txt", sep="\t", index=False)



In [18]:
#get full tax lineage for species level
dfm_subset = dfm[['best_taxid', 'phylum','class','order','family','genus','species']]
dfm_subset = dfm_subset.drop_duplicates(subset=['species'], keep='first')
dfm_species_tax = pd.merge(pivot_counts, dfm_subset, on = 'species', how = 'left')
dfm_species_tax.to_csv("culture/species_full_tax.txt", sep="\t", index=False)

